# K21 Academy | AWS Managed AI Services — Interactive Code Lab

An **interactive** code-based version of the AWS AI Services lab. Each section gives you buttons to click, status indicators, and live progress bars — no need to manually edit code or wait without feedback.

| Section | AWS Service | Interactive feature |
|---|---|---|
| 1 | Amazon Comprehend | Click a button to analyze each review one at a time |
| 2 | Amazon Translate | Real-time translation with status spinner |
| 3 | Amazon Transcribe | **Live progress bar** while the transcription job runs |
| 4 | Amazon Textract | Tabbed view: Raw text → Forms → Tables |
| Cleanup | S3 + Transcribe | One-click cleanup with confirmation |

---

### Prerequisites
1. AWS credentials configured (or running on SageMaker with execution role)
2. IAM permissions: `ComprehendFullAccess`, `TranslateFullAccess`, `AmazonTranscribeFullAccess`, `AmazonTextractFullAccess`, `AmazonS3FullAccess`
3. Sample files: **auto-downloaded** by the notebook — no manual setup

> Run cells **top-to-bottom**. Click the buttons inside each section to trigger the AWS calls.

## Setup — install & configure

In [ ]:
# Install dependencies
%pip install --quiet boto3 ipywidgets

In [ ]:
import json
import os
import time
import uuid
import urllib.request
from pathlib import Path

import boto3
from botocore.exceptions import ClientError
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
AWS_REGION = "us-east-1"
RUN_ID = uuid.uuid4().hex[:8]
BUCKET_NAME = f"k21-ai-lab-{RUN_ID}"
TRANSCRIBE_JOB_NAME = f"k21-sample-transcription-job-{RUN_ID}"

AUDIO_FILE_LOCAL = "transcribe-sample.mp3"
AUDIO_FILE_S3_KEY = "transcribe-sample.mp3"
DOCUMENT_FILE_LOCAL = "sample-document.png"

# Pre-create AWS clients (reused across sections)
comprehend = boto3.client("comprehend", region_name=AWS_REGION)
translate  = boto3.client("translate",  region_name=AWS_REGION)
transcribe = boto3.client("transcribe", region_name=AWS_REGION)
textract   = boto3.client("textract",   region_name=AWS_REGION)
s3         = boto3.client("s3",         region_name=AWS_REGION)

# Track which sections the learner completes (for the final scorecard idea)
progress = {"comprehend": False, "translate": False, "transcribe": False, "textract": False}

# Pretty status badges
def status_html(level, message):
    colors = {
        "info":    ("#1B9BD9", "ℹ️"),
        "success": ("#8DC63F", "✅"),
        "warn":    ("#FFA500", "⚠️"),
        "error":   ("#E74C3C", "❌"),
        "running": ("#1A1248", "⏳"),
    }
    color, icon = colors[level]
    return HTML(
        f'<div style="padding:10px;border-left:4px solid {color};'
        f'background:#f7f7fa;margin:6px 0;font-family:sans-serif;">'
        f'<b>{icon} {message}</b></div>'
    )

print(f"AWS region : {AWS_REGION}")
print(f"Run ID     : {RUN_ID}")
print(f"Bucket     : {BUCKET_NAME}")
print(f"Transcribe : {TRANSCRIBE_JOB_NAME}")
display(status_html("success", "Setup complete — clients ready"))

---
## Section 1 — Amazon Comprehend (Interactive)

Pick a review from the dropdown and click **Analyze**. The notebook will run all five Comprehend analyses (sentiment, entities, key phrases, language, syntax) and show the results below.

In [ ]:
reviews = {
    "Review 1 — Travel book (negative)": (
        "I just wanted to find some really cool new places such as Seattle in November. "
        "I've never visited before but no luck here. Some of these suggestions are just "
        "terrible... I had to laugh! Most suggestions were just your typical big cities, "
        "restaurants, and bars. Nothing off the beaten path here. I don't want to go to "
        "these places for fun. Totally not worth getting this."
    ),
    "Review 2 — Coffee table book (positive)": (
        "This was such a beautiful book. I wasn't even planning any travel when I came "
        "across this and just started flipping through the pages. I really like the cover "
        "and all the large glossy photographs in this book. John Smith did a wonderful "
        "job with the photography. I've found a perfect home for this on my coffee table. "
        "I'm planning a trip to Paris and Barcelona soon and I know this will come in handy."
    ),
    "Review 3 — Traveler review (positive)": (
        "As a traveler, I really appreciated reading about these great places to visit. "
        "The author takes you all over the world. Even with all the free information online "
        "these days, I find I'm taking this book with me wherever I go and using it to "
        "discover hidden gems."
    ),
    "Custom — Type your own": "",
}

# Widgets
review_dropdown = widgets.Dropdown(
    options=list(reviews.keys()),
    description="Review:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="500px"),
)
text_area = widgets.Textarea(
    value=reviews["Review 1 — Travel book (negative)"],
    placeholder="Or paste your own text here...",
    layout=widgets.Layout(width="100%", height="120px"),
)
analyze_button = widgets.Button(
    description="🔍 Analyze with Comprehend",
    button_style="primary",
    layout=widgets.Layout(width="280px"),
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=5,
    description="Progress:",
    bar_style="info",
    layout=widgets.Layout(width="500px"),
)
output_area = widgets.Output()

# When dropdown changes, update text area
def on_dropdown_change(change):
    if change["new"] != "Custom — Type your own":
        text_area.value = reviews[change["new"]]
    else:
        text_area.value = ""
review_dropdown.observe(on_dropdown_change, names="value")

# When the button is clicked, run all 5 analyses with progress updates
def on_analyze_click(b):
    output_area.clear_output()
    text = text_area.value.strip()
    if not text:
        with output_area:
            display(status_html("warn", "Please enter or pick some text first."))
        return

    analyze_button.disabled = True
    progress_bar.value = 0
    progress_bar.bar_style = "info"

    with output_area:
        display(status_html("running", "Calling Amazon Comprehend..."))

    try:
        # 1/5 Sentiment
        progress_bar.description = "Sentiment..."
        sentiment = comprehend.detect_sentiment(Text=text, LanguageCode="en")
        progress_bar.value = 1

        # 2/5 Entities
        progress_bar.description = "Entities..."
        entities = comprehend.detect_entities(Text=text, LanguageCode="en")
        progress_bar.value = 2

        # 3/5 Key phrases
        progress_bar.description = "Key phrases..."
        phrases = comprehend.detect_key_phrases(Text=text, LanguageCode="en")
        progress_bar.value = 3

        # 4/5 Language
        progress_bar.description = "Language..."
        lang = comprehend.detect_dominant_language(Text=text)
        progress_bar.value = 4

        # 5/5 Syntax
        progress_bar.description = "Syntax..."
        syntax = comprehend.detect_syntax(Text=text, LanguageCode="en")
        progress_bar.value = 5
        progress_bar.bar_style = "success"
        progress_bar.description = "Done ✓"

        # Render results
        output_area.clear_output()
        with output_area:
            display(status_html("success", "Analysis complete"))

            # Sentiment with emoji
            sent_emoji = {"POSITIVE": "😊", "NEGATIVE": "😞", "NEUTRAL": "😐", "MIXED": "🤔"}
            s = sentiment["Sentiment"]
            score = sentiment["SentimentScore"][s.title()]
            display(HTML(f"<h4>{sent_emoji.get(s,'')} Sentiment: <b>{s}</b> "
                         f"(confidence: {score:.1%})</h4>"))

            # Entities table
            if entities["Entities"]:
                rows = "".join(
                    f"<tr><td>{e['Text']}</td><td>{e['Type']}</td>"
                    f"<td>{e['Score']:.1%}</td></tr>"
                    for e in entities["Entities"][:10]
                )
                display(HTML(
                    f"<h4>🏷️ Entities ({len(entities['Entities'])} found)</h4>"
                    f"<table style='border-collapse:collapse;'>"
                    f"<tr style='background:#1A1248;color:white;'>"
                    f"<th style='padding:6px 12px;'>Text</th>"
                    f"<th style='padding:6px 12px;'>Type</th>"
                    f"<th style='padding:6px 12px;'>Score</th></tr>"
                    f"{rows}</table>"
                ))

            # Key phrases as chips
            if phrases["KeyPhrases"]:
                chips = " ".join(
                    f'<span style="background:#8DC63F;color:white;padding:3px 10px;'
                    f'border-radius:12px;margin:2px;display:inline-block;">{p["Text"]}</span>'
                    for p in phrases["KeyPhrases"][:8]
                )
                display(HTML(f"<h4>🔑 Key phrases</h4><div>{chips}</div>"))

            # Language
            top_lang = lang["Languages"][0]
            display(HTML(
                f"<h4>🌐 Dominant language: <b>{top_lang['LanguageCode']}</b> "
                f"({top_lang['Score']:.1%})</h4>"
            ))

            # Syntax POS counts
            pos_counts = {}
            for token in syntax["SyntaxTokens"]:
                tag = token["PartOfSpeech"]["Tag"]
                pos_counts[tag] = pos_counts.get(tag, 0) + 1
            top_pos = sorted(pos_counts.items(), key=lambda x: -x[1])[:5]
            display(HTML(
                f"<h4>📚 Top parts-of-speech</h4>"
                f"<div>{', '.join(f'<b>{t}</b>: {c}' for t, c in top_pos)}</div>"
            ))

        progress["comprehend"] = True

    except Exception as e:
        progress_bar.bar_style = "danger"
        progress_bar.description = "Failed"
        with output_area:
            display(status_html("error", f"{type(e).__name__}: {e}"))
    finally:
        analyze_button.disabled = False

analyze_button.on_click(on_analyze_click)

display(review_dropdown, text_area, analyze_button, progress_bar, output_area)

---
## Section 2 — Amazon Translate (Interactive)

Pick a source and target language, type or paste some text, then click **Translate**.

In [ ]:
LANGUAGES = {
    "auto (detect)": "auto", "English": "en", "French": "fr", "Spanish": "es",
    "German": "de", "Hindi": "hi", "Japanese": "ja", "Mandarin Chinese": "zh",
    "Arabic": "ar", "Portuguese": "pt", "Italian": "it", "Russian": "ru",
}

source_lang = widgets.Dropdown(options=list(LANGUAGES), value="French",
                                description="From:", style={"description_width": "60px"})
target_lang = widgets.Dropdown(options=list(LANGUAGES), value="English",
                                description="To:", style={"description_width": "60px"})

translate_input = widgets.Textarea(
    value=("Cette valise est d'un excellent rapport qualité prix et présente de très "
           "solides atouts. Elle est légère et parait solide. Néanmoins, elle présente "
           "un défaut TRES gênant: Rien n'est prévu pour la verrouiller."),
    placeholder="Type or paste text to translate...",
    layout=widgets.Layout(width="100%", height="100px"),
)
translate_button = widgets.Button(
    description="🌐 Translate",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)
translate_progress = widgets.IntProgress(
    value=0, min=0, max=1,
    description="Status:",
    bar_style="info",
    layout=widgets.Layout(width="500px"),
)
translate_output = widgets.Output()

def on_translate_click(b):
    translate_output.clear_output()
    text = translate_input.value.strip()
    if not text:
        with translate_output:
            display(status_html("warn", "Please enter some text to translate."))
        return

    translate_button.disabled = True
    translate_progress.value = 0
    translate_progress.bar_style = "info"
    translate_progress.description = "Calling..."

    with translate_output:
        display(status_html("running", "Calling Amazon Translate..."))

    try:
        response = translate.translate_text(
            Text=text,
            SourceLanguageCode=LANGUAGES[source_lang.value],
            TargetLanguageCode=LANGUAGES[target_lang.value],
        )
        translate_progress.value = 1
        translate_progress.bar_style = "success"
        translate_progress.description = "Done ✓"

        translate_output.clear_output()
        with translate_output:
            display(status_html("success", "Translation complete"))
            display(HTML(
                f"<table style='border-collapse:collapse;width:100%;'>"
                f"<tr><td style='vertical-align:top;padding:12px;background:#f7f7fa;width:50%;'>"
                f"<b>Source ({response['SourceLanguageCode']}):</b><br>{text}</td>"
                f"<td style='vertical-align:top;padding:12px;background:#e8f4ec;width:50%;'>"
                f"<b>Translation ({response['TargetLanguageCode']}):</b><br>"
                f"{response['TranslatedText']}</td></tr></table>"
            ))
        progress["translate"] = True

    except Exception as e:
        translate_progress.bar_style = "danger"
        translate_progress.description = "Failed"
        with translate_output:
            display(status_html("error", f"{type(e).__name__}: {e}"))
    finally:
        translate_button.disabled = False

translate_button.on_click(on_translate_click)

display(widgets.HBox([source_lang, target_lang]),
        translate_input, translate_button, translate_progress, translate_output)

---
## Section 3 — Amazon Transcribe (Interactive with Live Progress)

This section runs the **slowest** AWS service — a transcription job typically takes 30 seconds to 2 minutes. Click **Start Transcription** to:

1. Auto-download the sample MP3 (if not already present)
2. Create an S3 bucket and upload the audio
3. Start the transcription job
4. **Watch a live progress bar** while polling for completion
5. Display the transcript when done

In [ ]:
transcribe_button = widgets.Button(
    description="🎙️ Start Transcription",
    button_style="primary",
    layout=widgets.Layout(width="250px"),
)
# Indeterminate-style progress (we tick it forward each poll)
transcribe_progress = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Pipeline:",
    bar_style="info",
    layout=widgets.Layout(width="500px"),
)
transcribe_status = widgets.HTML(value="<i>Click the button to begin.</i>")
transcribe_output = widgets.Output()

def on_transcribe_click(b):
    transcribe_output.clear_output()
    transcribe_button.disabled = True
    transcribe_progress.bar_style = "info"

    def step(pct, msg):
        transcribe_progress.value = pct
        transcribe_status.value = f"<b>⏳ {msg}</b>"

    try:
        # Step 1 (10%) — download sample
        step(5, "Downloading sample audio file...")
        if not Path(AUDIO_FILE_LOCAL).exists():
            url = "https://github.com/k21academyuk/AWS-GenAI_AIML/raw/main/transcribe-sample.mp3"
            urllib.request.urlretrieve(url, AUDIO_FILE_LOCAL)
        step(10, f"Audio file ready ({Path(AUDIO_FILE_LOCAL).stat().st_size//1024} KB)")

        # Step 2 (25%) — create bucket
        step(15, "Creating S3 bucket...")
        try:
            if AWS_REGION == "us-east-1":
                s3.create_bucket(Bucket=BUCKET_NAME)
            else:
                s3.create_bucket(
                    Bucket=BUCKET_NAME,
                    CreateBucketConfiguration={"LocationConstraint": AWS_REGION},
                )
        except ClientError as err:
            if err.response["Error"]["Code"] not in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
                raise
        step(25, f"Bucket ready: s3://{BUCKET_NAME}")

        # Step 3 (40%) — upload
        step(30, "Uploading audio to S3...")
        s3.upload_file(AUDIO_FILE_LOCAL, BUCKET_NAME, AUDIO_FILE_S3_KEY)
        s3_uri = f"s3://{BUCKET_NAME}/{AUDIO_FILE_S3_KEY}"
        step(40, f"Uploaded to {s3_uri}")

        # Step 4 (50%) — start job
        step(45, "Starting transcription job...")
        transcribe.start_transcription_job(
            TranscriptionJobName=TRANSCRIBE_JOB_NAME,
            Media={"MediaFileUri": s3_uri},
            MediaFormat="mp3",
            LanguageCode="en-US",
        )
        step(50, f"Job started: {TRANSCRIBE_JOB_NAME}")

        # Step 5 (50% → 90%) — poll until COMPLETED, ticking forward each time
        poll_count = 0
        while True:
            time.sleep(5)
            poll_count += 1
            status_response = transcribe.get_transcription_job(
                TranscriptionJobName=TRANSCRIBE_JOB_NAME
            )
            job_status = status_response["TranscriptionJob"]["TranscriptionJobStatus"]
            # Move bar forward but cap at 89 so the user knows we're still polling
            tick_value = min(50 + poll_count * 5, 89)
            step(tick_value, f"Job status: {job_status} (poll #{poll_count})")
            if job_status in ("COMPLETED", "FAILED"):
                break

        if job_status == "FAILED":
            reason = status_response["TranscriptionJob"].get("FailureReason", "unknown")
            raise RuntimeError(f"Transcription failed: {reason}")

        # Step 6 (90% → 100%) — fetch transcript
        step(95, "Fetching transcript JSON...")
        transcript_uri = status_response["TranscriptionJob"]["Transcript"]["TranscriptFileUri"]
        with urllib.request.urlopen(transcript_uri) as resp:
            result = json.loads(resp.read().decode("utf-8"))
        transcript_text = result["results"]["transcripts"][0]["transcript"]

        transcribe_progress.value = 100
        transcribe_progress.bar_style = "success"
        transcribe_status.value = "<b>✅ Transcription complete</b>"

        with transcribe_output:
            display(HTML(
                f"<div style='padding:14px;border-left:4px solid #8DC63F;"
                f"background:#f7f7fa;font-family:sans-serif;'>"
                f"<h4 style='margin-top:0;'>📝 Transcript</h4>"
                f"<p style='line-height:1.6;'>{transcript_text}</p></div>"
            ))
        progress["transcribe"] = True

    except Exception as e:
        transcribe_progress.bar_style = "danger"
        transcribe_status.value = f"<b>❌ {type(e).__name__}: {e}</b>"
    finally:
        transcribe_button.disabled = False

transcribe_button.on_click(on_transcribe_click)
display(transcribe_button, transcribe_progress, transcribe_status, transcribe_output)

---
## Section 4 — Amazon Textract (Interactive with Tabs)

Click **Extract** to download a sample employment form and run it through Textract. Results are organized into tabs: **Raw Text**, **Forms**, and **Tables**.

In [ ]:
textract_button = widgets.Button(
    description="📄 Extract Text & Data",
    button_style="primary",
    layout=widgets.Layout(width="240px"),
)
textract_progress = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Pipeline:",
    bar_style="info",
    layout=widgets.Layout(width="500px"),
)
textract_status = widgets.HTML(value="<i>Click the button to begin.</i>")

# Three output panels - one per tab
raw_out    = widgets.Output()
forms_out  = widgets.Output()
tables_out = widgets.Output()
result_tabs = widgets.Tab(children=[raw_out, forms_out, tables_out])
result_tabs.set_title(0, "📝 Raw Text")
result_tabs.set_title(1, "📋 Forms")
result_tabs.set_title(2, "📊 Tables")

def on_extract_click(b):
    raw_out.clear_output(); forms_out.clear_output(); tables_out.clear_output()
    textract_button.disabled = True
    textract_progress.bar_style = "info"

    def step(pct, msg):
        textract_progress.value = pct
        textract_status.value = f"<b>⏳ {msg}</b>"

    try:
        # 1 — download sample doc
        step(10, "Downloading sample document...")
        if not Path(DOCUMENT_FILE_LOCAL).exists():
            url = ("https://raw.githubusercontent.com/aws-samples/"
                   "amazon-textract-code-samples/master/src-csharp/test-files/"
                   "employmentapp.png")
            urllib.request.urlretrieve(url, DOCUMENT_FILE_LOCAL)
        step(20, "Document ready, reading bytes...")

        with open(DOCUMENT_FILE_LOCAL, "rb") as f:
            doc_bytes = f.read()

        # 2 — OCR
        step(40, "Running OCR (DetectDocumentText)...")
        detect = textract.detect_document_text(Document={"Bytes": doc_bytes})

        # 3 — Forms + Tables
        step(70, "Running AnalyzeDocument (FORMS + TABLES)...")
        analyze = textract.analyze_document(
            Document={"Bytes": doc_bytes},
            FeatureTypes=["FORMS", "TABLES"],
        )

        step(90, "Rendering results...")

        # ---- Render Raw Text tab ----
        with raw_out:
            lines = [b["Text"] for b in detect["Blocks"] if b["BlockType"] == "LINE"]
            display(HTML(
                f"<h4>📝 {len(lines)} lines extracted</h4>"
                f"<pre style='background:#f7f7fa;padding:12px;max-height:400px;"
                f"overflow:auto;'>" + "\n".join(lines) + "</pre>"
            ))

        # Build a block map for FORMS / TABLES rendering
        blocks = analyze["Blocks"]
        block_map = {b["Id"]: b for b in blocks}

        def text_for(block):
            words = []
            for rel in block.get("Relationships", []):
                if rel["Type"] == "CHILD":
                    for cid in rel["Ids"]:
                        c = block_map[cid]
                        if c["BlockType"] == "WORD":
                            words.append(c["Text"])
                        elif c["BlockType"] == "SELECTION_ELEMENT":
                            words.append("[X]" if c["SelectionStatus"] == "SELECTED" else "[ ]")
            return " ".join(words)

        # ---- Render Forms tab ----
        with forms_out:
            kv_pairs = []
            for b in blocks:
                if b["BlockType"] == "KEY_VALUE_SET" and "KEY" in b.get("EntityTypes", []):
                    key_text = text_for(b)
                    value_text = ""
                    for rel in b.get("Relationships", []):
                        if rel["Type"] == "VALUE":
                            for vid in rel["Ids"]:
                                value_text = text_for(block_map[vid])
                    if key_text:
                        kv_pairs.append((key_text, value_text))

            if kv_pairs:
                rows = "".join(
                    f"<tr><td style='padding:6px 12px;border-bottom:1px solid #ddd;'><b>{k}</b></td>"
                    f"<td style='padding:6px 12px;border-bottom:1px solid #ddd;'>{v}</td></tr>"
                    for k, v in kv_pairs
                )
                display(HTML(
                    f"<h4>📋 {len(kv_pairs)} form fields detected</h4>"
                    f"<table style='border-collapse:collapse;'>"
                    f"<tr style='background:#1A1248;color:white;'>"
                    f"<th style='padding:8px 12px;'>Key</th>"
                    f"<th style='padding:8px 12px;'>Value</th></tr>"
                    f"{rows}</table>"
                ))
            else:
                display(HTML("<i>No form fields detected in this document.</i>"))

        # ---- Render Tables tab ----
        with tables_out:
            tables_found = [b for b in blocks if b["BlockType"] == "TABLE"]
            if not tables_found:
                display(HTML("<i>No tables detected in this document.</i>"))
            for ti, t in enumerate(tables_found, 1):
                cells_in_table = []
                for rel in t.get("Relationships", []):
                    if rel["Type"] == "CHILD":
                        for cid in rel["Ids"]:
                            c = block_map[cid]
                            if c["BlockType"] == "CELL":
                                cells_in_table.append(c)
                rows = {}
                for c in cells_in_table:
                    rows.setdefault(c["RowIndex"], {})[c["ColumnIndex"]] = text_for(c)
                html = f"<h4>📊 Table {ti}</h4><table style='border-collapse:collapse;'>"
                for row_idx in sorted(rows):
                    row = rows[row_idx]
                    bg = "#1A1248;color:white;" if row_idx == 1 else "white;"
                    cells_html = "".join(
                        f"<td style='padding:6px 12px;border:1px solid #ddd;background:{bg}'>"
                        f"{row[col]}</td>" for col in sorted(row)
                    )
                    html += f"<tr>{cells_html}</tr>"
                html += "</table>"
                display(HTML(html))

        textract_progress.value = 100
        textract_progress.bar_style = "success"
        textract_status.value = "<b>✅ Extraction complete — switch tabs to explore</b>"
        progress["textract"] = True

    except Exception as e:
        textract_progress.bar_style = "danger"
        textract_status.value = f"<b>❌ {type(e).__name__}: {e}</b>"
    finally:
        textract_button.disabled = False

textract_button.on_click(on_extract_click)
display(textract_button, textract_progress, textract_status, result_tabs)

---
## Cleanup — Delete S3 Bucket and Transcribe Job

Click **Cleanup Resources** to remove everything created during the lab. A confirmation step prevents accidental deletion.

In [ ]:
confirm_checkbox = widgets.Checkbox(
    value=False,
    description="I confirm — delete the bucket and transcribe job",
    indent=False,
    layout=widgets.Layout(width="500px"),
)
cleanup_button = widgets.Button(
    description="🗑️ Cleanup Resources",
    button_style="danger",
    layout=widgets.Layout(width="220px"),
)
cleanup_progress = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Cleanup:",
    bar_style="info",
    layout=widgets.Layout(width="500px"),
)
cleanup_output = widgets.Output()

def on_cleanup_click(b):
    cleanup_output.clear_output()
    if not confirm_checkbox.value:
        with cleanup_output:
            display(status_html("warn", "Tick the confirmation box first."))
        return
    cleanup_button.disabled = True
    cleanup_progress.value = 0
    cleanup_progress.bar_style = "info"

    with cleanup_output:
        # Delete transcribe job
        try:
            transcribe.delete_transcription_job(TranscriptionJobName=TRANSCRIBE_JOB_NAME)
            display(status_html("success", f"Deleted transcribe job: {TRANSCRIBE_JOB_NAME}"))
        except ClientError as err:
            display(status_html("warn",
                f"Transcribe cleanup skipped: {err.response['Error']['Code']}"))
        cleanup_progress.value = 50

        # Empty + delete bucket
        try:
            objects = s3.list_objects_v2(Bucket=BUCKET_NAME).get("Contents", [])
            if objects:
                s3.delete_objects(
                    Bucket=BUCKET_NAME,
                    Delete={"Objects": [{"Key": o["Key"]} for o in objects]},
                )
                display(status_html("success", f"Emptied bucket: s3://{BUCKET_NAME}"))
            s3.delete_bucket(Bucket=BUCKET_NAME)
            display(status_html("success", f"Deleted bucket: s3://{BUCKET_NAME}"))
        except ClientError as err:
            display(status_html("warn",
                f"S3 cleanup skipped: {err.response['Error']['Code']}"))

        cleanup_progress.value = 100
        cleanup_progress.bar_style = "success"
        display(status_html("success", "Cleanup complete — lab finished 🎉"))

cleanup_button.on_click(on_cleanup_click)
display(confirm_checkbox, cleanup_button, cleanup_progress, cleanup_output)